# Python Match Statement

> **Python Mastery** · Module 02 — Control Flow · Lesson 2/5

`match` is Python's answer to the switch statement - but smarter. Instead of comparing one value against constants, it can match the *shape* of your data: lists, dictionaries, whole structures. It arrived in **Python 3.10**, so check your version before using it.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Write** `match`/`case` blocks and explain how they differ from C-style `switch`
- **Combine** literal values with the or-pattern (`|`) and catch-all with `_`
- **Bind** matched data to variables with capture patterns
- **Add** guards (`case n if n > 10:`) for extra conditions
- **Match** the structure of lists, tuples, and dictionaries
- **Decide** when a plain `if...elif` chain is the better tool

## 1. First: Check Your Python Version

Structural pattern matching was added in **Python 3.10** (PEP 634). On older interpreters the word `match` is just an ordinary name and the syntax is a `SyntaxError`. One quick cell tells you where you stand:

In [1]:
import sys

print("Running Python", ".".join(map(str, sys.version_info[:2])))
assert sys.version_info >= (3, 10), "match/case needs Python 3.10+"
print("match/case available")

Running Python 3.14
match/case available


## 2. Basic Syntax: `match` / `case`

Python takes an expression, tries each `case` pattern top-down, and runs **the first one that fits**. A wildcard `_` at the bottom plays the role of `else`.

Two big differences from C/Java `switch`:

- **No fall-through.** Exactly one case body runs; there is no `break`, and control never leaks into the next case.
- **No constant-only restriction** - patterns can destructure data, as you'll see below.

**Syntax:**
```python
match value:
    case pattern_1:
        ...   # value fits pattern_1
    case pattern_2:
        ...
    case _:   # wildcard: matches anything, like else
        ...
```

In [2]:
def status_message(code):
    match code:
        case 200:
            return "OK"
        case 301:
            return "Moved Permanently"
        case 404:
            return "Not Found"
        case 500:
            return "Server Error - our fault"
        case _:
            return f"Unknown status {code}"

for code in (200, 404, 418):
    print(code, "->", status_message(code))

200 -> OK
404 -> Not Found
418 -> Unknown status 418


## 3. Combining Values with `|`

Several literals that share the same handling can sit in one case, joined by `|`. Read it as *"or"*.

**Syntax:**
```python
match command:
    case "start" | "go":
        ...
```

In [3]:
command = "go"

match command:
    case "start" | "go" | "run":
        print("Robot moving forward")
    case "stop" | "halt" | "quit":
        print("Robot stopping")
    case _:
        print(f"Unrecognized command: {command}")

Robot moving forward


## 4. The Wildcard `_`

`_` matches **anything** and binds nothing. Put it last as your safety net.

> ⚠️ If no case matches *and* there is no `_`, Python does nothing silently - no error, no warning. Always ask yourself what should happen when nothing fits.

In [4]:
http_method = "DELETE"

match http_method:
    case "GET" | "HEAD":
        print("Read-only request")
    case "POST" | "PUT" | "PATCH":
        print("Writes data")
    case _:                       # everything else lands here
        print("Method not handled:", http_method)

Method not handled: DELETE


## 5. Capture Patterns: Binding the Matched Value

A bare variable name in a pattern **captures** whatever matched - it doesn't compare against that name. That's how you get the unmatched value into your hands:

**Syntax:**
```python
match value:
    case number:      # captures ANY single value into `number`
        ...
```

> 🔍 **Under the Hood:** How does Python know `case number:` should capture rather than compare with an existing variable `number`? Rule: a **dotted** name (like `Color.RED` or `MAX_SIZE`) is treated as a *value* to compare against; any other bare name is a *capture*. So `case MAX_RETRIES:` compares with the module-level constant, while `case max_retries:` would silently capture - a classic source of bugs when constants are lowercase.

In [ ]:
def describe(x):
    match x:
        case 0:
            return "zero"
        case 1 | -1:
            return "plus or minus one"
        case other:               # captures anything left over
            return f"something else: {other!r}"

for sample in (0, -1, "banana", 3.14):
    print(describe(sample))

# The classic trap:
LIMIT = 100
value = 42
match value:
    case LIMIT:                    # dotted? No -> plain name -> CAPTURES!
        pass
print("After 'case LIMIT:', LIMIT was rebound to:", LIMIT)

zero
plus or minus one
something else: 'banana'
something else: 3.14
After 'case LIMIT:', LIMIT was rebound to: 42


## 6. Guards: Adding Conditions with `if`

A pattern answers *"does it look like this?"*; a guard answers *"and is this also true?"*. Attach it after the pattern with `if`. If the guard fails, matching simply continues to later cases.

**Syntax:**
```python
case pattern if condition:
    ...
```

In [6]:
weight_kg = 12

shipping = None
match weight_kg:
    case n if n <= 0:
        shipping = "invalid weight"
    case n if n <= 1:
        shipping = 60       # BDT
    case n if n <= 5:
        shipping = 120
    case n if n <= 10:
        shipping = 200
    case _:
        shipping = "quote required"

print(f"Shipping for {weight_kg} kg: {shipping}")

# Guards can inspect parts of a structured match too - see next sections.

Shipping for 12 kg: quote required


## 7. Matching Sequences: Lists and Tuples

This is where `match` leaves `switch` far behind. A list/tuple pattern checks both the **shape** (how many elements?) and the **contents** (what's in which slot?), then binds names for you.

**Syntax:**
```python
match point:
    case (0, 0):        # exactly these values
        ...
    case (0, y):        # first item 0, second captured as y
        ...
    case [first, *rest]:  # at least one item; rest gathered into a list
        ...
```

In [7]:
points = [(0, 0), (0, 7), (3, 0), (2, 9)]

for point in points:
    match point:
        case (0, 0):
            print(point, "-> the origin")
        case (0, y):
            print(point, f"-> on the y-axis, height {y}")
        case (x, 0):
            print(point, f"-> on the x-axis, position {x}")
        case (x, y):
            print(point, f"-> general point ({x}, {y})")
# Same idea on lists, with a star to soak up extras:

(0, 0) -> the origin
(0, 7) -> on the y-axis, height 7
(3, 0) -> on the x-axis, position 3
(2, 9) -> general point (2, 9)


In [8]:
scores = [95, 88, 72, 61]

match scores:
    case []:
        print("Empty leaderboard")
    case [top]:
        print(f"Solo leader with {top}")
    case [top, second, *others]:
        print(f"Gold {top}, silver {second}, plus {len(others)} more players")
# Change scores to [95] and re-run: the shape itself picks the branch.

Gold 95, silver 88, plus 2 more players


## 8. Matching Dictionaries (Mapping Patterns)

Dict patterns check only the keys you **list** - extra keys are fine and ignored. This makes them perfect for pulling fields out of JSON-ish records without writing `record["..."]` five times.

**Syntax:**
```python
match record:
    case {"role": "admin", "name": who}:
        ...   # role must equal 'admin'; name is captured
```

In [9]:
users = [
    {"name": "Sarah", "role": "admin", "city": "Dhaka"},
    {"name": "Rahim", "role": "member"},
    {"city": "Chattogram"},
]

for user in users:
    match user:
        case {"role": "admin", "name": who}:
            print(f"Admin detected: {who} - full access")
        case {"role": role, "name": who}:
            print(f"{who} is a {role}")
        case _:
            print(f"Incomplete profile: {user}")
# Note: {'name': ..., 'role': ..., 'city': ...} still matched the admin
# pattern even though 'city' was never mentioned - extra keys are ignored.

Admin detected: Sarah - full access
Rahim is a member
Incomplete profile: {'city': 'Chattogram'}


> 🔍 **Under the Hood:** This is why the feature is called ***structural* pattern matching** (PEP 634). Old-school `switch` asks *"is the value equal to X?"* - a flat equality test. Python's `match` instead asks *"does the value have this **structure**?"* - it recursively walks your data, comparing literals where you wrote literals and binding names wherever you wrote names. `(0, y)` isn't a tuple being compared with `==`; it's a *pattern* describing a two-element sequence whose first slot is zero. The same machinery extends to your own classes: `case Point(x=0, y=0):` works once the class declares `__match_args__`, letting you pattern-match objects just like tuples and dicts.

## 10. When Plain `if...elif` Is Still Better

`match` shines when one subject has many possible *shapes or values*. Reach for `if...elif` instead when:

- You're testing **ranges/thresholds** with simple booleans (`if temp > 30:`) - guards can do it, but they read worse.
- The conditions involve **several different variables**, not variations of one value.
- You need **complex boolean logic** (`and`/`or` combinations across fields).
- Your team ships to Python versions **older than 3.10**.

Rule of thumb: `match` = *dispatching*, `if` = *deciding*.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| No wildcard and nothing matches | Runs silently - no error, easy to miss | End with `case _:` (even just logging) |
| `case existing_var:` expecting comparison | Bare name *captures* and overwrites your variable | Use a dotted/value form (`case MyEnum.YES:`) or a literal |
| `case age > 65:` | `SyntaxError` - comparisons aren't allowed in pattern position | Use a guard: `case age if age > 65:` |
| Expecting fall-through like C `switch` | Each case is fully independent; there's no deliberate falling into the next case | List shared values together with `\|` |
| Forgetting Python < 3.10 support | `SyntaxError` on older interpreters | Guard with `sys.version_info`, or stick to if/elif |

In [10]:
role = "editor"

# WRONG intent: author thought this compares role to the variable 'admin_user'
admin_user = "admin"
match role:
    case admin_user:              # captures! always matches!
        print(f"bug: bound admin_user={admin_user!r}, role was {role!r}")

# RIGHT: compare via a guard or restructure the data
match role:
    case r if r == admin_user:
        print("correct: real admin")
    case _:
        print(f"correct: '{role}' is not an admin")

bug: bound admin_user='editor', role was 'editor'
correct: real admin


## 💡 Best Practices & Pro Tips

- **Order cases from most specific to most general** - the first structural fit wins, so put exact shapes (`(0, 0)`) above loose ones (`(x, y)`).
- **Always provide a `case _:`** so unexpected data fails loudly or logs, rather than vanishing.
- **Use guards sparingly** - if half your cases need `if`, the problem wants plain `if...elif`.
- **Prefer mapping patterns over chains of `data.get(...)`** when unpacking API responses; the pattern documents the expected shape right in the header.
- **AI-engineering relevance:** LLM/agent pipelines dispatch constantly on message *shapes* - e.g. routing `{"type": "tool_call", ...}` vs `{"type": "final_answer", ...}` events, or handling streaming payloads that may arrive as different JSON skeletons. Structural matching turns those nested key checks into one readable block.

## 📌 Summary

| Pattern | What it does | Example |
|---|---|---|
| Literal | equals that exact value | `case 404:` |
| Or-pattern `\|` | any of several values | `case "go" \| "run":` |
| Wildcard `_` | matches anything, binds nothing | `case _:` |
| Capture `name` | binds the matched value | `case other:` |
| Guard `if cond` | extra boolean filter on a case | `case n if n > 10:` |
| Sequence `[a, b]` / `(x, y)` | matches list/tuple shape, binds slots | `case [top, *rest]:` |
| Mapping `{"k": v}` | matches listed dict keys, ignores extras | `case {"role": "admin", "name": who}:` |

Key takeaways:

- `match` needs **Python 3.10+** and has **no fall-through** - one case, one execution.
- Patterns match **structure**, not just equality; that's why it's called structural pattern matching.
- A bare name captures instead of comparing - the #1 surprise for newcomers.
- Dispatching on one value's many shapes → `match`; deciding across many variables → `if`.

## 🔗 Next Lesson

Up next: **[03_While_Loop](../03_While_Loop/notes.ipynb)** - repeating work with `while`, plus `break`, `continue`, and the sneaky loop-`else` clause.